In [ ]:
!apt-get -qq update
!apt-get -qq install -y git-lfs
!git lfs install
!pip -q install tqdm


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Failed to fetch https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu/dists/jammy/InRelease  Could not connect to ppa.launchpadcontent.net:443 (185.125.190.80), connection timed out
W: Failed to fetch https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu/dists/jammy/InRelease  Unable to connect to ppa.launchpadcontent.net:443:
W: Some index files failed to download. They have been ignored, or old ones used instead.
Git LFS initialized.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/openai/prm800k.git"
REPO_DIR = Path("/content/prm800k")

if not REPO_DIR.exists():
    !git clone {REPO_URL} /content/prm800k
else:
    print("Repo already exists at /content/prm800k")

%cd /content/prm800k
!git lfs pull

print("\nData directory contents:")
!ls -lh /content/prm800k/prm800k/data


Cloning into '/content/prm800k'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 28 (delta 1), reused 0 (delta 0), pack-reused 20 (from 1)
Receiving objects: 100% (28/28), 4.86 MiB | 12.91 MiB/s, done.
Resolving deltas: 100% (2/2), done.
Filtering content: 100% (6/6), 465.82 MiB | 7.70 MiB/s, done.
/content/prm800k

Data directory contents:
total 456M
-rw-r--r-- 1 root root 810K May  3 01:48 phase1_test.jsonl
-rw-r--r-- 1 root root 7.6M May  3 01:48 phase1_train.jsonl
-rw-r--r-- 1 root root  12M May  3 01:48 phase2_test.jsonl
-rw-r--r-- 1 root root 436M May  3 01:49 phase2_train.jsonl


In [ ]:
INPUT_FILES = [
    "/content/prm800k/prm800k/data/phase2_test.jsonl"
]

OUTPUT_DIR = "/content/prm800k_processed"
SEED = 42


In [ ]:
import json
import random
from pathlib import Path
from typing import Any, Dict, List
from tqdm.auto import tqdm


def normalize_text(x: Any) -> str:
    if x is None:
        return ""
    return str(x).strip()


def read_jsonl(path: str):
    with open(path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError as e:
                print(f"Skipping malformed JSON in {path} line {line_num}: {e}")


def write_jsonl(path: str, rows: List[Dict[str, Any]]):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def join_steps(steps: List[str]) -> str:
    cleaned = [normalize_text(s) for s in steps if normalize_text(s)]
    return "\n".join(cleaned)


def reconstruct_chosen_prefixes(labeled_steps: List[Dict[str, Any]]) -> List[List[str]]:
    prefixes: List[List[str]] = []
    current_prefix: List[str] = []

    for step in labeled_steps:
        prefixes.append(list(current_prefix))

        chosen_idx = step.get("chosen_completion")
        if chosen_idx is None:
            human_completion = step.get("human_completion")
            if isinstance(human_completion, dict):
                chosen_text = normalize_text(human_completion.get("text"))
            else:
                chosen_text = normalize_text(human_completion)
        else:
            completions = step.get("completions", [])
            if isinstance(completions, list) and 0 <= chosen_idx < len(completions):
                chosen_text = normalize_text(completions[chosen_idx].get("text"))
            else:
                chosen_text = ""

        if chosen_text:
            current_prefix.append(chosen_text)

    return prefixes


rng = random.Random(SEED)
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

next_step_rows: List[Dict[str, Any]] = []
positives_by_problem: Dict[str, Dict[str, Any]] = {}
negative_candidates_by_problem: Dict[str, List[Dict[str, Any]]] = {}

num_samples = 0
for input_path in INPUT_FILES:
    print(f"Processing {input_path} ...")
    for sample in tqdm(read_jsonl(input_path)):
        num_samples += 1

        question = sample.get("question", {})
        label_obj = sample.get("label", {})

        problem = normalize_text(question.get("problem"))
        ground_truth_solution = normalize_text(question.get("ground_truth_solution"))
        pre_generated_steps = question.get("pre_generated_steps", None)
        labeled_steps = label_obj.get("steps", [])
        finish_reason = normalize_text(label_obj.get("finish_reason"))

        if not problem:
            continue

        if isinstance(labeled_steps, list):
            prefixes = reconstruct_chosen_prefixes(labeled_steps)

            for step_index, step in enumerate(labeled_steps):
                completions = step.get("completions", [])
                if not isinstance(completions, list):
                    continue

                prefix_steps = prefixes[step_index]

                for completion in completions:
                    target_step = normalize_text(completion.get("text"))
                    raw_rating = completion.get("rating")
                    flagged = completion.get("flagged")

                    if not target_step:
                        continue
                    if flagged is True:
                        continue
                    if raw_rating not in (-1, 0, 1):
                        continue

                    next_step_rows.append({
                        "instruction": "Verify whether the target step is valid given the problem and the previous steps.",
                        "problem": problem,
                        "prefix_steps": prefix_steps,
                        "target_step": target_step,
                        "label": 1 if raw_rating in (0, 1) else 0,
                        "raw_rating": raw_rating,
                        "step_index": step_index,
                    })

        if problem not in positives_by_problem and ground_truth_solution:
            positives_by_problem[problem] = {
                "instruction": "Verify whether the following full solution is correct.",
                "problem": problem,
                "solution": ground_truth_solution,
                "label": 1,
            }

        if finish_reason == "found_error" and isinstance(pre_generated_steps, list) and len(pre_generated_steps) > 0:
            solution_text = join_steps(pre_generated_steps)
            if solution_text:
                negative_candidates_by_problem.setdefault(problem, []).append({
                    "instruction": "Verify whether the following full solution is correct.",
                    "problem": problem,
                    "solution": solution_text,
                    "label": 0,
                })

solution_rows: List[Dict[str, Any]] = []
all_problems = sorted(set(positives_by_problem.keys()) | set(negative_candidates_by_problem.keys()))

for problem in all_problems:
    if problem in positives_by_problem:
        solution_rows.append(positives_by_problem[problem])
    if problem in negative_candidates_by_problem:
        solution_rows.append(rng.choice(negative_candidates_by_problem[problem]))

next_step_out = output_dir / "next_step_verification.jsonl"
solution_out = output_dir / "solution_verification.jsonl"

write_jsonl(str(next_step_out), next_step_rows)
write_jsonl(str(solution_out), solution_rows)

print("\nDone.")
print(f"Processed {num_samples} PRM800K rows")
print(f"Wrote {next_step_out} with {len(next_step_rows)} rows")
print(f"Wrote {solution_out} with {len(solution_rows)} rows")
print(f"Unique problems in solution dataset: {len(set(r['problem'] for r in solution_rows))}")
print(f"Solution positives: {sum(r['label'] == 1 for r in solution_rows)}")
print(f"Solution negatives: {sum(r['label'] == 0 for r in solution_rows)}")

if next_step_rows:
    print("\nExample next-step row:")
    print(json.dumps(next_step_rows[0], indent=2, ensure_ascii=False)[:1500])

if solution_rows:
    print("\nExample solution-verification row:")
    print(json.dumps(solution_rows[0], indent=2, ensure_ascii=False)[:1500])


Processing /content/prm800k/prm800k/data/phase2_test.jsonl ...


0it [00:00, ?it/s]


Done.
Processed 2762 PRM800K rows
Wrote /content/prm800k_processed/next_step_verification.jsonl with 26256 rows
Wrote /content/prm800k_processed/solution_verification.jsonl with 905 rows
Unique problems in solution dataset: 458
Solution positives: 458
Solution negatives: 447

Example next-step row:
{
  "instruction": "Verify whether the target step is valid given the problem and the previous steps.",
  "problem": "A Senate committee has 5 Democrats, 5 Republicans, and 1 Independent.  In how many ways can they sit around a circular table if all the members of each party all sit next to each other?  (Two seatings are considered equivalent if one is a rotation of the other.)",
  "prefix_steps": [],
  "target_step": "I notice that there are three groups of people: Democrats, Republicans, and Independent.",
  "label": 1,
  "raw_rating": 1,
  "step_index": 0
}

Example solution-verification row:
{
  "instruction": "Verify whether the following full solution is correct.",
  "problem": "$441+

In [ ]:
from google.colab import files
files.download(str(next_step_out))
files.download(str(solution_out))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>